# Equal and spatially gated fusion without auxiliary supervision

Seed 42. Measured equal-fusion parameter count is 7,932,769; the later runner has a stale approximate expected-count constant, documented separately.

Source: `graindiscoloration (2).ipynb`, zero-based cell 46.

Run in a fresh Kaggle GPU session with the appropriate dataset attached. Review the configuration near the top; original dataset/output paths are retained. Do not Run All across the repository.


In [ ]:
"""
GD-RGBMS-DB — SIX-CHANNEL COMPONENT ABLATION — SEED 42 — NO TTA
=================================================================

One-cell Kaggle/Colab script that trains two controlled variants in order:

  A1. Equal multi-scale fusion
      - separate RGB and spectral encoders
      - fixed 0.5 / 0.5 fusion at four scales
      - progressive cross-stage feedback
      - no auxiliary heads or auxiliary loss

  A2. Spatially gated multi-scale fusion
      - separate RGB and spectral encoders
      - learned per-pixel two-way softmax gate at four scales
      - progressive cross-stage feedback
      - no auxiliary heads or auxiliary loss

Both runs use the same six original channels (RGB, Green, Red-Edge, NIR),
provided 280/60/60 split, 896x672 inputs, seed 42, GroupNorm, optimizer,
loss, augmentation, training schedule, validation-only threshold selection,
and single-pass no-TTA test evaluation as the final six-channel model.

Paste this entire file into ONE Kaggle notebook cell and run it. The
train+validation RAM cache is built once and shared by both experiments.
"""

from __future__ import annotations

import csv
import gc
import hashlib
import json
import os
import random
import shutil
import time
from pathlib import Path

import numpy as np
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset


# =============================================================================
# 1. CONFIGURATION
# =============================================================================

# Set this manually only if automatic discovery selects the wrong copy.
# BASE_DIR = "/kaggle/input/datasets/hemal7/gd-rgbms-db"
# BASE_DIR = "/kaggle/input/datasets/jarin19/rgbmsdb"
# BASE_DIR = "/content/GD-RGBMS-DB"
BASE_DIR = None

SEED = 42
TARGET_SIZE = (896, 672)  # width, height
BATCH_SIZE = 2
ACCUMULATION_STEPS = 2
MAX_EPOCHS = 70
PATIENCE = 12
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 0

BRANCH_BASE = 28
PREFERRED_GN_GROUPS = 8
CACHE_TRAIN_AND_VALIDATION = True
SAVE_TEST_PREDICTIONS = False

# These counts act as architecture guards. The spatial model count equals the
# deployment count of the final six-channel model after removing its two
# training-only auxiliary heads.
EXPECTED_PARAMETER_COUNTS = {
    "equal": 7_932_769,
    "spatial": 7_950_033,
}

# Run both sequentially. Do not change the order for the paper experiment.
VARIANTS = (
    {
        "key": "equal_multiscale",
        "display_name": "A1 Equal Multi-Scale Fusion",
        "fusion_type": "equal",
    },
    {
        "key": "spatial_gated",
        "display_name": "A2 Spatially Gated Multi-Scale Fusion",
        "fusion_type": "spatial",
    },
)

MASK_POSITIVE_RGB = np.array([240, 128, 128], dtype=np.int16)
RGB_CHANNELS = 3
SPECTRAL_CHANNELS = 3
TOTAL_INPUT_CHANNELS = 6
INPUT_CHANNEL_ORDER = ("R", "G", "B", "Green", "Red-Edge", "NIR")

# Training-split statistics used by the final six-channel experiment.
MEAN = np.array(
    [
        0.41819797,
        0.46573568,
        0.25167707,
        0.21822916,
        0.16206901,
        0.23585168,
    ],
    dtype=np.float32,
)
STD = np.array(
    [
        0.15404519,
        0.15691774,
        0.13375558,
        0.07147331,
        0.05124885,
        0.04640943,
    ],
    dtype=np.float32,
)

if len(MEAN) != TOTAL_INPUT_CHANNELS or len(STD) != TOTAL_INPUT_CHANNELS:
    raise ValueError("MEAN and STD must each contain six channel values.")
if np.any(STD <= 0):
    raise ValueError("All channel standard deviations must be positive.")

TRAIN_THRESHOLDS = np.round(np.arange(0.20, 0.86, 0.05), 2).tolist()
FINAL_THRESHOLDS = np.round(np.arange(0.20, 0.801, 0.01), 2).tolist()

if Path("/kaggle/working").exists():
    WORK_DIR = Path("/kaggle/working")
elif Path("/content").exists():
    WORK_DIR = Path("/content")
else:
    WORK_DIR = Path.cwd()

OUT_ROOT = WORK_DIR / "gd_6ch_component_ablation_seed42"
OUT_ROOT.mkdir(parents=True, exist_ok=True)

try:
    BILINEAR = Image.Resampling.BILINEAR
    NEAREST = Image.Resampling.NEAREST
except AttributeError:
    BILINEAR = Image.BILINEAR
    NEAREST = Image.NEAREST


# =============================================================================
# 2. DATASET DISCOVERY, SPLIT AUDIT, AND REPRODUCIBILITY
# =============================================================================

def read_list(path):
    with open(path, "r", encoding="utf-8") as file:
        return [line.strip() for line in file if line.strip()]


def valid_root(path):
    required = (
        path / "RGB",
        path / "MS",
        path / "Mark",
        path / "train_list.txt",
        path / "val_list.txt",
        path / "test_list.txt",
    )
    if not all(item.exists() for item in required):
        return False
    return (
        len(read_list(path / "train_list.txt")) == 280
        and len(read_list(path / "val_list.txt")) == 60
        and len(read_list(path / "test_list.txt")) == 60
    )


def discover_dataset_roots():
    roots = [Path("/kaggle/input"), Path("/content"), Path("/content/drive/MyDrive")]
    candidates = []
    for root in roots:
        if not root.exists():
            continue
        for item in root.rglob("train_list.txt"):
            if valid_root(item.parent):
                candidates.append(item.parent)
    unique = sorted(set(candidates), key=lambda p: ("gd-rgbms" not in str(p).lower(), len(str(p))))
    return unique


def split_digest(root):
    digest = hashlib.sha256()
    for filename in ("train_list.txt", "val_list.txt", "test_list.txt"):
        names = read_list(root / filename)
        digest.update(filename.encode("utf-8"))
        digest.update("\n".join(names).encode("utf-8"))
    return digest.hexdigest()


if BASE_DIR is None:
    roots = discover_dataset_roots()
    if not roots:
        raise FileNotFoundError(
            "Could not find a GD-RGBMS-DB copy with the provided 280/60/60 "
            "split. Set BASE_DIR manually near the top of the cell."
        )
    BASE_DIR = roots[0]
    if len(roots) > 1:
        digests = {str(root): split_digest(root) for root in roots}
        identical = len(set(digests.values())) == 1
        print("Detected dataset copies:")
        for root, digest in digests.items():
            print(" ", root, "| split SHA256:", digest)
        print("Split lists identical across detected copies:", identical)
        if not identical:
            raise RuntimeError(
                "Multiple dataset copies have different split lists. Set "
                "BASE_DIR explicitly to the copy used by the final model."
            )

BASE_DIR = Path(BASE_DIR)
if not valid_root(BASE_DIR):
    raise FileNotFoundError(f"Invalid dataset root: {BASE_DIR}")


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


seed_everything(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pin_memory = device.type == "cuda"

train_names = read_list(BASE_DIR / "train_list.txt")
val_names = read_list(BASE_DIR / "val_list.txt")
test_names = read_list(BASE_DIR / "test_list.txt")

split_sets = [set(train_names), set(val_names), set(test_names)]
if any(
    len(names) != len(unique_names)
    for names, unique_names in zip(
        (train_names, val_names, test_names), split_sets
    )
):
    raise ValueError("Duplicate filenames exist within a dataset split.")
if (
    split_sets[0] & split_sets[1]
    or split_sets[0] & split_sets[2]
    or split_sets[1] & split_sets[2]
):
    raise ValueError("Exact filename overlap exists across dataset splits.")

rgb_dir = BASE_DIR / "RGB"
ms_dir = BASE_DIR / "MS"
mark_dir = BASE_DIR / "Mark"

print("=" * 80)
print("SIX-CHANNEL COMPONENT ABLATION — SEED 42 — NO TTA")
print("=" * 80)
print("Dataset       :", BASE_DIR)
print("Split digest  :", split_digest(BASE_DIR))
print("Output        :", OUT_ROOT)
print("Device        :", device)
if torch.cuda.is_available():
    print("GPU           :", torch.cuda.get_device_name(0))
print("Split         :", len(train_names), len(val_names), len(test_names))
print("Input size    :", TARGET_SIZE, "(width, height)")
print("Channels      :", list(INPUT_CHANNEL_ORDER))
print("Batch/accum   :", BATCH_SIZE, "/", ACCUMULATION_STEPS)
print("Normalization : GroupNorm")
print("Runs          :", " -> ".join(item["display_name"] for item in VARIANTS))


# =============================================================================
# 3. PREPROCESSING AND SHARED RAM CACHE
# =============================================================================

def find_path(directory, stem, extensions):
    for extension in extensions:
        path = directory / f"{stem}{extension}"
        if path.exists():
            return path
    raise FileNotFoundError(f"Could not find {stem} in {directory}")


def binary_mask(image):
    array = np.asarray(image.convert("RGB"), dtype=np.int16)
    return (np.abs(array - MASK_POSITIVE_RGB).sum(axis=2) < 80).astype(np.float32)


def load_raw(name):
    stem = Path(name).stem
    rgb_path = find_path(rgb_dir, stem, [".JPG", ".jpg", ".JPEG", ".jpeg"])
    ms_path = find_path(ms_dir, stem, [".PNG", ".png"])
    mark_path = find_path(mark_dir, stem, [".png", ".PNG"])

    with Image.open(rgb_path) as item:
        rgb_image = item.convert("RGB")
    with Image.open(ms_path) as item:
        ms_image = item.convert("RGB")
    with Image.open(mark_path) as item:
        mark_image = item.convert("RGB")

    rgb_image = rgb_image.resize(TARGET_SIZE, BILINEAR)
    ms_image = ms_image.resize(TARGET_SIZE, BILINEAR)
    mark_image = mark_image.resize(TARGET_SIZE, NEAREST)

    rgb = np.asarray(rgb_image, dtype=np.float32).copy() / 255.0
    ms = np.asarray(ms_image, dtype=np.float32).copy() / 255.0
    mask = binary_mask(mark_image)
    image = np.concatenate([rgb, ms], axis=2)
    if image.shape[2] != TOTAL_INPUT_CHANNELS:
        raise RuntimeError(
            f"Expected six channels but received {image.shape[2]} for {stem}."
        )
    return image.astype(np.float32), mask, stem


RAM_CACHE = {}


def preload_cache(names):
    if not CACHE_TRAIN_AND_VALIDATION:
        return
    for name in tqdm(names, desc="Preloading shared train+val RAM cache"):
        image, mask, stem = load_raw(name)
        RAM_CACHE[name] = (
            image.astype(np.float16),
            mask.astype(np.uint8),
            stem,
        )
    approximate_gb = sum(
        image.nbytes + mask.nbytes
        for image, mask, _ in RAM_CACHE.values()
    ) / (1024**3)
    print(f"RAM cache: {len(RAM_CACHE)} images, {approximate_gb:.2f} GB")


preload_cache(train_names + val_names)


class GDDataset(Dataset):
    def __init__(self, names, augment=False):
        self.names = list(names)
        self.augment = augment

    def __len__(self):
        return len(self.names)

    def __getitem__(self, index):
        name = self.names[index]
        if name in RAM_CACHE:
            cached_image, cached_mask, stem = RAM_CACHE[name]
            image = cached_image.astype(np.float32)
            mask = cached_mask.astype(np.float32)
        else:
            image, mask, stem = load_raw(name)

        if self.augment:
            if random.random() < 0.5:
                image = np.flip(image, axis=1).copy()
                mask = np.flip(mask, axis=1).copy()
            if random.random() < 0.5:
                image = np.flip(image, axis=0).copy()
                mask = np.flip(mask, axis=0).copy()
            if random.random() < 0.75:
                gain = np.random.uniform(0.88, 1.12, (1, 1, 3)).astype(np.float32)
                bias = np.random.uniform(-0.035, 0.035, (1, 1, 3)).astype(np.float32)
                image[..., :3] = np.clip(image[..., :3] * gain + bias, 0.0, 1.0)

        image = ((image - MEAN) / STD).astype(np.float32)
        return (
            torch.from_numpy(image.transpose(2, 0, 1)).float(),
            torch.from_numpy(mask[None]).float(),
            stem,
        )


def make_loaders(seed):
    generator = torch.Generator()
    generator.manual_seed(seed)
    train_loader = DataLoader(
        GDDataset(train_names, augment=True),
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=generator,
        num_workers=NUM_WORKERS,
        pin_memory=pin_memory,
        drop_last=False,
    )
    val_loader = DataLoader(
        GDDataset(val_names),
        batch_size=1,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=pin_memory,
    )
    test_loader = DataLoader(
        GDDataset(test_names),
        batch_size=1,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=pin_memory,
    )
    return train_loader, val_loader, test_loader


# =============================================================================
# 4. SHARED ARCHITECTURE
# =============================================================================

def group_norm(channels):
    groups = min(PREFERRED_GN_GROUPS, channels)
    while channels % groups:
        groups -= 1
    return nn.GroupNorm(groups, channels, eps=1e-5, affine=True)


class SCSE(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        hidden = max(channels // reduction, 1)
        self.channel = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(channels, hidden, 1),
            nn.SiLU(inplace=True),
            nn.Conv2d(hidden, channels, 1),
            nn.Sigmoid(),
        )
        self.spatial = nn.Sequential(nn.Conv2d(channels, 1, 1), nn.Sigmoid())

    def forward(self, x):
        return x * self.channel(x) + x * self.spatial(x)


class ResBlock(nn.Module):
    def __init__(self, in_channels, out_channels, dropout=0.0):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False),
            group_norm(out_channels),
            nn.SiLU(inplace=True),
            nn.Dropout2d(dropout) if dropout else nn.Identity(),
            nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
            group_norm(out_channels),
        )
        self.skip = (
            nn.Identity()
            if in_channels == out_channels
            else nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1, bias=False),
                group_norm(out_channels),
            )
        )
        self.attention = SCSE(out_channels)

    def forward(self, x):
        return self.attention(F.silu(self.block(x) + self.skip(x), inplace=True))


class EqualFusion(nn.Module):
    """Fixed 0.5/0.5 fusion with the same post-fusion refinement path."""

    def __init__(self, channels):
        super().__init__()
        self.refine = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1, bias=False),
            group_norm(channels),
            nn.SiLU(inplace=True),
        )

    def forward(self, rgb_feature, spectral_feature):
        fused = 0.5 * rgb_feature + 0.5 * spectral_feature
        return self.refine(fused)


class SpatialGatedFusion(nn.Module):
    """Two learned weights sum to one at every spatial location."""

    def __init__(self, channels):
        super().__init__()
        hidden = max(channels // 8, 8)
        self.gate = nn.Sequential(
            nn.Conv2d(channels * 2, hidden, 1, bias=False),
            group_norm(hidden),
            nn.SiLU(inplace=True),
            nn.Conv2d(hidden, 2, 1),
        )
        self.refine = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1, bias=False),
            group_norm(channels),
            nn.SiLU(inplace=True),
        )

    def forward(self, rgb_feature, spectral_feature):
        weights = torch.softmax(
            self.gate(torch.cat([rgb_feature, spectral_feature], dim=1)), dim=1
        )
        fused = (
            weights[:, 0:1] * rgb_feature
            + weights[:, 1:2] * spectral_feature
        )
        return self.refine(fused)


class UpBlock(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels, dropout=0.0):
        super().__init__()
        self.conv = ResBlock(in_channels + skip_channels, out_channels, dropout)

    def forward(self, x, skip):
        x = F.interpolate(
            x, size=skip.shape[-2:], mode="bilinear", align_corners=False
        )
        return self.conv(torch.cat([x, skip], dim=1))


class SixChannelMultiScaleFusionUNet(nn.Module):
    def __init__(self, fusion_type, base=28):
        super().__init__()
        if fusion_type not in {"equal", "spatial"}:
            raise ValueError(f"Unknown fusion type: {fusion_type}")
        self.fusion_type = fusion_type
        c1, c2, c3, c4 = [base, base * 2, base * 4, base * 8]
        self.pool = nn.MaxPool2d(2)

        self.rgb1 = ResBlock(RGB_CHANNELS, c1)
        self.spec1 = ResBlock(SPECTRAL_CHANNELS, c1)
        self.rgb2, self.spec2 = ResBlock(c1, c2), ResBlock(c1, c2)
        self.rgb3, self.spec3 = ResBlock(c2, c3, 0.05), ResBlock(c2, c3, 0.05)
        self.rgb4, self.spec4 = ResBlock(c3, c4, 0.10), ResBlock(c3, c4, 0.10)

        fusion_class = EqualFusion if fusion_type == "equal" else SpatialGatedFusion
        self.fuse1 = fusion_class(c1)
        self.fuse2 = fusion_class(c2)
        self.fuse3 = fusion_class(c3)
        self.fuse4 = fusion_class(c4)

        self.center = ResBlock(c4, c4 * 2, 0.10)
        self.up4 = UpBlock(c4 * 2, c4, c4, 0.10)
        self.up3 = UpBlock(c4, c3, c3, 0.05)
        self.up2 = UpBlock(c3, c2, c2)
        self.up1 = UpBlock(c2, c1, c1)
        self.output = nn.Conv2d(c1, 1, 1)

    def forward(self, x):
        if x.ndim != 4 or x.shape[1] != TOTAL_INPUT_CHANNELS:
            raise ValueError(
                f"Expected [B, 6, H, W], received {tuple(x.shape)}."
            )
        rgb = x[:, :RGB_CHANNELS]
        spectral = x[:, RGB_CHANNELS:]

        r1, s1 = self.rgb1(rgb), self.spec1(spectral)
        f1 = self.fuse1(r1, s1)

        # Progressive cross-stage feedback is identical in both variants.
        r2 = self.rgb2(self.pool(r1 + f1))
        s2 = self.spec2(self.pool(s1 + f1))
        f2 = self.fuse2(r2, s2)

        r3 = self.rgb3(self.pool(r2 + f2))
        s3 = self.spec3(self.pool(s2 + f2))
        f3 = self.fuse3(r3, s3)

        r4 = self.rgb4(self.pool(r3 + f3))
        s4 = self.spec4(self.pool(s3 + f3))
        f4 = self.fuse4(r4, s4)

        center = self.center(self.pool(f4))
        d4 = self.up4(center, f4)
        d3 = self.up3(d4, f3)
        d2 = self.up2(d3, f2)
        d1 = self.up1(d2, f1)
        return self.output(d1)


# =============================================================================
# 5. LOSS AND METRICS
# =============================================================================

class ComboLoss(nn.Module):
    def forward(self, logits, targets):
        targets = targets.float()
        bce_map = F.binary_cross_entropy_with_logits(
            logits, targets, reduction="none"
        )
        bce = bce_map.mean()
        probabilities = torch.sigmoid(logits)
        pt = probabilities * targets + (1 - probabilities) * (1 - targets)
        focal = (bce_map * ((1 - pt) ** 2.0)).mean()
        dims = tuple(range(1, logits.ndim))
        intersection = (probabilities * targets).sum(dims)
        denominator = probabilities.sum(dims) + targets.sum(dims)
        dice_loss = 1 - (
            (2 * intersection + 1e-6) / (denominator + 1e-6)
        ).mean()
        return 0.35 * bce + 0.25 * focal + 0.40 * dice_loss


criterion = ComboLoss()


def confusion(probabilities, masks, threshold):
    if torch.is_tensor(probabilities):
        prediction = probabilities >= threshold
        target = masks >= 0.5
        return [
            int((prediction & target).sum().item()),
            int((prediction & ~target).sum().item()),
            int((~prediction & target).sum().item()),
            int((~prediction & ~target).sum().item()),
        ]
    prediction = probabilities >= threshold
    target = masks >= 0.5
    return [
        int(np.logical_and(prediction, target).sum()),
        int(np.logical_and(prediction, np.logical_not(target)).sum()),
        int(np.logical_and(np.logical_not(prediction), target).sum()),
        int(np.logical_and(np.logical_not(prediction), np.logical_not(target)).sum()),
    ]


def metrics(values):
    tp, fp, fn, tn = values
    epsilon = 1e-7
    return {
        "dice": (2 * tp + epsilon) / (2 * tp + fp + fn + epsilon),
        "iou": (tp + epsilon) / (tp + fp + fn + epsilon),
        "precision": (tp + epsilon) / (tp + fp + epsilon),
        "recall": (tp + epsilon) / (tp + fn + epsilon),
        "accuracy": (tp + tn + epsilon) / (tp + fp + fn + tn + epsilon),
    }


def save_csv(path, rows):
    if not rows:
        return
    with open(path, "w", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(file, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)


def safe_load(path):
    try:
        return torch.load(path, map_location=device, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=device)


# =============================================================================
# 6. ONE CONTROLLED RUN
# =============================================================================

def run_variant(specification):
    variant_key = specification["key"]
    display_name = specification["display_name"]
    fusion_type = specification["fusion_type"]
    out_dir = OUT_ROOT / variant_key
    out_dir.mkdir(parents=True, exist_ok=True)

    # Reset every random source so the two runs use the same seed and data order.
    seed_everything(SEED)
    train_loader, val_loader, test_loader = make_loaders(SEED)
    model = SixChannelMultiScaleFusionUNet(
        fusion_type=fusion_type, base=BRANCH_BASE
    ).to(device)
    parameter_count = sum(parameter.numel() for parameter in model.parameters())
    expected_count = EXPECTED_PARAMETER_COUNTS[fusion_type]
    if parameter_count != expected_count:
        raise RuntimeError(
            f"Architecture mismatch for {fusion_type}: expected "
            f"{expected_count:,} parameters, found {parameter_count:,}."
        )

    # Lightweight shape/finite-value check before a long training run starts.
    model.eval()
    with torch.no_grad():
        sample = torch.zeros(1, TOTAL_INPUT_CHANNELS, 64, 64, device=device)
        sample_output = model(sample)
    if sample_output.shape != (1, 1, 64, 64):
        raise RuntimeError(
            f"Unexpected model output shape: {tuple(sample_output.shape)}."
        )
    if not torch.isfinite(sample_output).all():
        raise RuntimeError("The architecture preflight produced non-finite values.")
    del sample, sample_output

    optimizer = torch.optim.AdamW(
        model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", factor=0.5, patience=4, min_lr=1e-6
    )
    scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")

    checkpoint_path = out_dir / f"best_{variant_key}_6ch_seed42.pt"
    history_path = out_dir / "training_history.csv"
    history = []
    best_dice = -1.0
    best_epoch = 0

    print("\n" + "=" * 80)
    print(display_name.upper())
    print("=" * 80)
    print("Fusion type          :", fusion_type)
    print("Cross-stage feedback : enabled")
    print("Auxiliary supervision: disabled")
    print("Trainable parameters :", f"{parameter_count:,}")

    def train_epoch():
        model.train()
        optimizer.zero_grad(set_to_none=True)
        running_loss = 0.0
        for step, (images, masks, _) in enumerate(
            tqdm(train_loader, desc=f"{variant_key} training", leave=False), start=1
        ):
            images = images.to(device, non_blocking=True)
            masks = masks.to(device, non_blocking=True)
            with torch.amp.autocast("cuda", enabled=device.type == "cuda"):
                logits = model(images)
                loss = criterion(logits, masks)
                scaled_loss = loss / ACCUMULATION_STEPS
            scaler.scale(scaled_loss).backward()
            running_loss += loss.item()
            if step % ACCUMULATION_STEPS == 0 or step == len(train_loader):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
        return running_loss / len(train_loader)

    @torch.no_grad()
    def validate_coarse():
        model.eval()
        totals = {threshold: [0, 0, 0, 0] for threshold in TRAIN_THRESHOLDS}
        loss_total = 0.0
        for images, masks, _ in tqdm(
            val_loader, desc=f"{variant_key} validation", leave=False
        ):
            images = images.to(device, non_blocking=True)
            masks = masks.to(device, non_blocking=True)
            with torch.amp.autocast("cuda", enabled=device.type == "cuda"):
                logits = model(images)
                loss_total += criterion(logits, masks).item()
            probabilities = torch.sigmoid(logits)
            for threshold in TRAIN_THRESHOLDS:
                values = confusion(probabilities, masks, threshold)
                totals[threshold] = [
                    old + new for old, new in zip(totals[threshold], values)
                ]
        rows = []
        for threshold, values in totals.items():
            row = metrics(values)
            row.update(
                {
                    "threshold": float(threshold),
                    "loss": loss_total / len(val_loader),
                }
            )
            rows.append(row)
        return max(rows, key=lambda item: item["dice"])

    for epoch in range(1, MAX_EPOCHS + 1):
        start = time.time()
        train_loss = train_epoch()
        validation = validate_coarse()
        scheduler.step(validation["dice"])
        minutes = (time.time() - start) / 60.0
        learning_rate = optimizer.param_groups[0]["lr"]
        history.append(
            {
                "epoch": epoch,
                "train_loss": train_loss,
                "val_loss": validation["loss"],
                "val_dice": validation["dice"],
                "val_iou": validation["iou"],
                "val_precision": validation["precision"],
                "val_recall": validation["recall"],
                "val_accuracy": validation["accuracy"],
                "val_threshold": validation["threshold"],
                "learning_rate": learning_rate,
                "minutes": minutes,
            }
        )
        print(
            f"{variant_key:17s} | Epoch {epoch:03d} | "
            f"train {train_loss:.4f} | val {validation['loss']:.4f} | "
            f"Dice {validation['dice']:.4f} | IoU {validation['iou']:.4f} | "
            f"P {validation['precision']:.4f} | R {validation['recall']:.4f} | "
            f"th {validation['threshold']:.2f} | lr {learning_rate:.2e} | "
            f"{minutes:.1f}m"
        )

        if validation["dice"] > best_dice:
            best_dice = validation["dice"]
            best_epoch = epoch
            torch.save(
                {
                    "model": model.state_dict(),
                    "epoch": epoch,
                    "best_coarse_validation_dice": best_dice,
                    "coarse_validation_threshold": validation["threshold"],
                    "variant_key": variant_key,
                    "display_name": display_name,
                    "fusion_type": fusion_type,
                    "spatial_gate": fusion_type == "spatial",
                    "cross_stage_feedback": True,
                    "auxiliary_supervision": False,
                    "seed": SEED,
                    "target_size": TARGET_SIZE,
                    "mean": MEAN.tolist(),
                    "std": STD.tolist(),
                    "input_channel_order": list(INPUT_CHANNEL_ORDER),
                    "derived_ndre_included": False,
                    "normalization": "GroupNorm",
                    "branch_base": BRANCH_BASE,
                    "parameter_count": parameter_count,
                },
                checkpoint_path,
            )
            print("Saved:", checkpoint_path)

        if epoch - best_epoch >= PATIENCE:
            print(f"Early stopping after {PATIENCE} non-improving epochs.")
            break

    save_csv(history_path, history)
    checkpoint = safe_load(checkpoint_path)
    model.load_state_dict(checkpoint["model"], strict=True)
    model.eval()

    @torch.no_grad()
    def collect_predictions(loader, description):
        records = []
        for images, masks, stems in tqdm(loader, desc=description):
            images = images.to(device, non_blocking=True)
            with torch.amp.autocast("cuda", enabled=device.type == "cuda"):
                probabilities = torch.sigmoid(model(images))
            records.append(
                {
                    "stem": stems[0],
                    "mask": masks[0, 0].numpy().astype(np.float32),
                    "probability": probabilities[0, 0]
                    .float()
                    .cpu()
                    .numpy()
                    .astype(np.float32),
                }
            )
        return records

    def threshold_search(records, thresholds):
        rows = []
        for threshold in thresholds:
            total = [0, 0, 0, 0]
            for record in records:
                values = confusion(
                    record["probability"], record["mask"], threshold
                )
                total = [old + new for old, new in zip(total, values)]
            row = metrics(total)
            row["threshold"] = float(threshold)
            rows.append(row)
        return max(rows, key=lambda item: item["dice"]), rows

    def evaluate_records(records, threshold):
        global_values = [0, 0, 0, 0]
        per_image_rows = []
        for record in records:
            values = confusion(record["probability"], record["mask"], threshold)
            global_values = [
                old + new for old, new in zip(global_values, values)
            ]
            row = metrics(values)
            row.update(
                {
                    "stem": record["stem"],
                    "threshold": threshold,
                    "tp": values[0],
                    "fp": values[1],
                    "fn": values[2],
                    "tn": values[3],
                }
            )
            per_image_rows.append(row)
        result = metrics(global_values)
        result.update(
            {
                "threshold": threshold,
                "tp": global_values[0],
                "fp": global_values[1],
                "fn": global_values[2],
                "tn": global_values[3],
            }
        )
        return result, per_image_rows

    validation_records = collect_predictions(
        val_loader, f"{variant_key} final validation predictions (no TTA)"
    )
    selected_validation, threshold_rows = threshold_search(
        validation_records, FINAL_THRESHOLDS
    )
    selected_threshold = selected_validation["threshold"]
    save_csv(out_dir / "validation_threshold_search_no_tta.csv", threshold_rows)
    save_csv(
        out_dir / "selected_validation_metrics_no_tta.csv",
        [selected_validation],
    )
    del validation_records

    test_records = collect_predictions(
        test_loader, f"{variant_key} final test predictions (no TTA)"
    )
    test_metrics, per_image_rows = evaluate_records(
        test_records, selected_threshold
    )
    save_csv(out_dir / "primary_no_tta_test_metrics.csv", [test_metrics])
    save_csv(out_dir / "per_image_test_metrics_no_tta.csv", per_image_rows)

    if SAVE_TEST_PREDICTIONS:
        prediction_dir = out_dir / "test_predictions_no_tta"
        probability_dir = out_dir / "test_probability_uint16_no_tta"
        prediction_dir.mkdir(exist_ok=True)
        probability_dir.mkdir(exist_ok=True)
        for record in tqdm(test_records, desc=f"Saving {variant_key} outputs"):
            prediction = (
                record["probability"] >= selected_threshold
            ).astype(np.uint8) * 255
            probability = np.clip(
                record["probability"] * 65535.0, 0, 65535
            ).astype(np.uint16)
            Image.fromarray(prediction).save(
                prediction_dir / f"{record['stem']}.png"
            )
            Image.fromarray(probability).save(
                probability_dir / f"{record['stem']}.png"
            )

    summary = {
        "model": display_name,
        "variant_key": variant_key,
        "fusion_type": fusion_type,
        "fixed_equal_weights": [0.5, 0.5] if fusion_type == "equal" else None,
        "spatial_gate": fusion_type == "spatial",
        "cross_stage_feedback": True,
        "auxiliary_supervision": False,
        "dataset_root": str(BASE_DIR),
        "split_digest_sha256": split_digest(BASE_DIR),
        "seed": SEED,
        "split_sizes": {
            "train": len(train_names),
            "validation": len(val_names),
            "test": len(test_names),
        },
        "input_channels": list(INPUT_CHANNEL_ORDER),
        "include_ndre": False,
        "target_size_width_height": list(TARGET_SIZE),
        "normalization_layer": "GroupNorm",
        "normalization_mean": MEAN.tolist(),
        "normalization_std": STD.tolist(),
        "batch_size": BATCH_SIZE,
        "accumulation_steps": ACCUMULATION_STEPS,
        "epochs_requested": MAX_EPOCHS,
        "epochs_completed": len(history),
        "best_epoch": checkpoint["epoch"],
        "early_stopping_patience": PATIENCE,
        "optimizer": "AdamW",
        "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "scheduler": "ReduceLROnPlateau(mode=max,factor=0.5,patience=4,min_lr=1e-6)",
        "loss": "0.35 BCE + 0.25 focal(gamma=2) + 0.40 Dice",
        "parameter_count": parameter_count,
        "checkpoint_selected_by": "highest coarse-grid validation Dice",
        "threshold_selected_by": "highest fine-grid validation Dice",
        "selected_threshold": selected_threshold,
        "test_time_augmentation": "none",
        "validation_metrics": selected_validation,
        "primary_no_tta_test_metrics": test_metrics,
    }
    with open(out_dir / "run_summary.json", "w", encoding="utf-8") as file:
        json.dump(summary, file, indent=2)

    checkpoint["selected_validation_metrics_no_tta"] = selected_validation
    checkpoint["selected_threshold"] = selected_threshold
    checkpoint["primary_no_tta_test_metrics"] = test_metrics
    torch.save(checkpoint, checkpoint_path)

    variant_zip = shutil.make_archive(
        str(WORK_DIR / f"gd_6ch_{variant_key}_seed42_results"),
        "zip",
        str(out_dir),
    )

    print("\n" + "-" * 80)
    print(display_name, "— FINAL PRIMARY TEST RESULT — NO TTA")
    print("-" * 80)
    print("Dice      :", f"{test_metrics['dice']:.4f}")
    print("IoU       :", f"{test_metrics['iou']:.4f}")
    print("Precision :", f"{test_metrics['precision']:.4f}")
    print("Recall    :", f"{test_metrics['recall']:.4f}")
    print("Threshold :", f"{selected_threshold:.2f}")
    print("Parameters:", f"{parameter_count / 1e6:.3f} M")
    print("Best epoch:", checkpoint["epoch"])
    print("Results   :", out_dir)
    print("ZIP       :", variant_zip)

    concise_result = {
        "configuration": display_name,
        "fusion_type": fusion_type,
        "spatial_gate": fusion_type == "spatial",
        "cross_stage_feedback": True,
        "auxiliary_supervision": False,
        "parameters": parameter_count,
        "best_epoch": checkpoint["epoch"],
        **test_metrics,
        "results_zip": variant_zip,
    }

    del test_records, per_image_rows
    del model, optimizer, scheduler, scaler
    del train_loader, val_loader, test_loader
    gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()
    return concise_result


# =============================================================================
# 7. RUN BOTH VARIANTS AND PACKAGE ALL RESULTS
# =============================================================================

all_results = []
for variant in VARIANTS:
    all_results.append(run_variant(variant))

combined_rows = []
for result in all_results:
    combined_rows.append(
        {
            "configuration": result["configuration"],
            "fusion_type": result["fusion_type"],
            "spatial_gate": result["spatial_gate"],
            "cross_stage_feedback": result["cross_stage_feedback"],
            "auxiliary_supervision": result["auxiliary_supervision"],
            "parameters": result["parameters"],
            "best_epoch": result["best_epoch"],
            "dice": result["dice"],
            "iou": result["iou"],
            "precision": result["precision"],
            "recall": result["recall"],
            "accuracy": result["accuracy"],
            "threshold": result["threshold"],
        }
    )

combined_csv = OUT_ROOT / "six_channel_component_ablation_summary.csv"
save_csv(combined_csv, combined_rows)

combined_json = OUT_ROOT / "six_channel_component_ablation_summary.json"
with open(combined_json, "w", encoding="utf-8") as file:
    json.dump(
        {
            "dataset_root": str(BASE_DIR),
            "split_digest_sha256": split_digest(BASE_DIR),
            "seed": SEED,
            "input_channels": list(INPUT_CHANNEL_ORDER),
            "include_ndre": False,
            "test_time_augmentation": "none",
            "results": all_results,
        },
        file,
        indent=2,
    )

combined_zip = shutil.make_archive(
    str(WORK_DIR / "gd_6ch_component_ablation_seed42_results"),
    "zip",
    str(OUT_ROOT),
)

print("\n" + "=" * 80)
print("BOTH SIX-CHANNEL COMPONENT ABLATION RUNS FINISHED")
print("=" * 80)
for row in combined_rows:
    print(
        f"{row['configuration']}: "
        f"Dice={row['dice']:.4f} | IoU={row['iou']:.4f} | "
        f"P={row['precision']:.4f} | R={row['recall']:.4f} | "
        f"th={row['threshold']:.2f}"
    )
print("Combined CSV :", combined_csv)
print("Combined JSON:", combined_json)
print("Combined ZIP :", combined_zip)
print("IMPORTANT: Download the ZIP before ending the Kaggle session.")

try:
    from IPython.display import FileLink, display

    display(FileLink(combined_zip))
    for result in all_results:
        display(FileLink(result["results_zip"]))
except Exception as error:
    print("Inline download links could not be displayed:", error)